In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.1 Characters as tokens

> 🎯 **Goal:** Turn raw text into a sequence of integers (and back) using the simplest possible scheme: one integer per character.

**Why this matters.** A language model only does math on numbers, never on letters. Before any embedding, attention, or training can happen, text has to become a list of integers called token ids. The character tokenizer is the gentlest first step, and PocketLM starts here so you can see the whole text-to-numbers pipeline with nothing hidden.

**The intuition.** Think of a secret decoder ring where every letter has a number stamped next to it: A is 1, B is 2, and so on. Encoding is just reading the numbers off the ring; decoding is reading the letters back. Because the mapping is fixed and total, the round trip is lossless for any character the ring knows.

**The mechanics.** A *token* is the atomic unit the model sees. A *vocabulary* is the set of all tokens, and its size is the *vocab size*. `CharTokenizer.from_text(corpus)` scans the corpus, collects every distinct character, and assigns each one a stable integer id. `encode` replaces each character with its id; `decode` does the reverse lookup. There is no training and no statistics, just a one-to-one table between characters and integers.

In [ ]:
from pocketlm import CharTokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
print("vocab size:", tok.vocab_size)
ids = tok.encode("To be")
print(ids, "->", repr(tok.decode(ids)))

**What just happened.** The print showed the vocab size (how many distinct characters the corpus contains) and a round trip: `"To be"` became a short list of ids, and decoding those ids gave `"To be"` back exactly. For text built only from characters the tokenizer has seen, encode-then-decode is perfectly lossless.

⚠️ **Common pitfalls**
- Out-of-vocabulary characters: if you encode a character the tokenizer never saw during `from_text` (say an accented letter or an emoji absent from the corpus), it is not in the table and the round trip breaks. Character tokenizers have holes.
- Long sequences: one token per character means a 1000-character paragraph becomes 1000 tokens. Since attention cost grows with sequence length, this gets expensive fast. This length cost is exactly what the next lesson attacks.

🏋️ **Try it yourself.** Change `"To be"` to a string that contains a character not in the corpus (try an emoji like the smiley, or a rare symbol) and rerun. Watch the round trip fail or raise, then confirm a plain ASCII sentence still round-trips cleanly.

In [ ]:
assert tok.decode(tok.encode("To be")) == "To be"